# Phase 2 — Synthetic Data Generation

## Step 2.2 — Site Master Data

### Objective
Create the master 'site' dataset for the Intelligent Equipment Operations Hub.

The project will simulate three operational sites. Each site will later contain equipment assets, sensor readings, maintenance events, failures, work orders, and associated costs.

### Output
`sites.csv`

### Step 1 — Create the sites Dataset

In [1]:
import pandas as pd

sites = pd.DataFrame({
    "Site_ID": [
        "SITE001",
        "SITE002",
        "SITE003"
    ],
    "Site_Name": [
        "Belfast Operations Plant",
        "Larne Equipment Centre",
        "Dublin Operations Hub"
    ],
    "Location": [
        "Belfast",
        "Larne",
        "Dublin"
    ],
    "Region": [
        "Northern Ireland",
        "Northern Ireland",
        "Ireland"
    ],
    "Site_Type": [
        "Manufacturing",
        "Industrial",
        "Distribution"
    ],
    "Operational_Status": [
        "Active",
        "Active",
        "Active"
    ],
    "Commission_Date": pd.to_datetime([
        "2018-04-15",
        "2020-09-01",
        "2022-03-10"
    ])
})

sites

,Site_ID,Site_Name,Location,Region,Site_Type,Operational_Status,Commission_Date
0,SITE001,Belfast Operations Plant,Belfast,Northern Ireland,Manufacturing,Active,2018-04-15
1,SITE002,Larne Equipment Centre,Larne,Northern Ireland,Industrial,Active,2020-09-01
2,SITE003,Dublin Operations Hub,Dublin,Ireland,Distribution,Active,2022-03-10


In [2]:
print("Rows:", len(sites))
print("Duplicate Site_ID:", sites["Site_ID"].duplicated().sum())
print("\nMissing values:")
print(sites.isnull().sum())

Rows: 3
Duplicate Site_ID: 0

Missing values:
Site_ID               0
Site_Name             0
Location              0
Region                0
Site_Type             0
Operational_Status    0
Commission_Date       0
dtype: int64


In [ ]:
## Save the file

""" sites.to_csv("../data/sites.csv", index=False)
print("sites.csv created successfully.") """

sites.csv created successfully.


### Step 2— Asset Master Data

### Objective
Generate 150 equipment assets across the three operational sites.

- 50 assets per site
- Realistic equipment categories
- Unique asset and serial numbers
- Installation dates, criticality, capacity, and operating status

### Output
`assets.csv`

In [4]:
import numpy as np

np.random.seed(42)

# -----------------------------
# Configuration
# -----------------------------

asset_types = {
    "Compressor": {
        "manufacturer": ["Atlas Copco", "Ingersoll Rand"],
        "models": ["GA30", "GA45", "R-Series"],
        "capacity_range": (20, 60),
        "unit": "kW"
    },
    "Pump": {
        "manufacturer": ["Grundfos", "KSB"],
        "models": ["CR15", "CR20", "Etanorm"],
        "capacity_range": (50, 250),
        "unit": "m3/h"
    },
    "Motor": {
        "manufacturer": ["Siemens", "ABB"],
        "models": ["SIMOTICS GP", "M3BP"],
        "capacity_range": (10, 90),
        "unit": "kW"
    },
    "Fan": {
        "manufacturer": ["ABB", "FläktGroup"],
        "models": ["AXIAL-500", "CENT-750"],
        "capacity_range": (5000, 20000),
        "unit": "m3/h"
    },
    "Conveyor": {
        "manufacturer": ["SEW-Eurodrive", "Dorner"],
        "models": ["MOVIMOT", "3200 Series"],
        "capacity_range": (500, 2500),
        "unit": "kg/h"
    }
}

In [5]:
site_ids = sites["Site_ID"].tolist()

records = []

asset_counter = 1

for site_id in site_ids:
    
    for _ in range(50):
        
        asset_type = np.random.choice(list(asset_types.keys()))
        config = asset_types[asset_type]

        manufacturer = np.random.choice(config["manufacturer"])
        model = np.random.choice(config["models"])

        install_date = pd.Timestamp("2015-01-01") + pd.to_timedelta(
            np.random.randint(0, 3650), unit="D"
        )

        criticality = np.random.choice(
            ["High", "Medium", "Low"],
            p=[0.30, 0.45, 0.25]
        )

        asset_status = np.random.choice(
            ["Active", "Maintenance", "Offline"],
            p=[0.90, 0.07, 0.03]
        )

        capacity = round(
            np.random.uniform(
                config["capacity_range"][0],
                config["capacity_range"][1]
            ),
            2
        )

        expected_life = np.random.choice([10, 12, 15, 20])

        asset_id = f"AST{asset_counter:04d}"

        serial_number = f"SN-{asset_counter:06d}"

        warranty_end_date = install_date + pd.DateOffset(years=5)

        records.append({
            "Asset_ID": asset_id,
            "Site_ID": site_id,
            "Asset_Name": f"{asset_type} {asset_counter:03d}",
            "Asset_Type": asset_type,
            "Manufacturer": manufacturer,
            "Model": model,
            "Serial_Number": serial_number,
            "Install_Date": install_date,
            "Criticality": criticality,
            "Rated_Capacity": capacity,
            "Capacity_Unit": config["unit"],
            "Asset_Status": asset_status,
            "Expected_Life_Years": expected_life,
            "Warranty_End_Date": warranty_end_date
        })

        asset_counter += 1

assets = pd.DataFrame(records)

assets.head()

,Asset_ID,Site_ID,Asset_Name,Asset_Type,Manufacturer,Model,Serial_Number,Install_Date,Criticality,Rated_Capacity,Capacity_Unit,Asset_Status,Expected_Life_Years,Warranty_End_Date
0,AST0001,SITE001,Fan 001,Fan,ABB,AXIAL-500,SN-000001,2018-02-04,Low,11687.49,m3/h,Active,15,2023-02-04
1,AST0002,SITE001,Motor 002,Motor,Siemens,M3BP,SN-000002,2024-06-06,Medium,11.65,kW,Active,12,2029-06-06
2,AST0003,SITE001,Fan 003,Fan,FläktGroup,CENT-750,SN-000003,2021-08-30,High,9563.63,m3/h,Active,12,2026-08-30
3,AST0004,SITE001,Conveyor 004,Conveyor,Dorner,MOVIMOT,SN-000004,2024-12-11,Medium,593.33,kg/h,Active,20,2029-12-11
4,AST0005,SITE001,Motor 005,Motor,ABB,SIMOTICS GP,SN-000005,2018-06-21,Medium,58.60,kW,Active,10,2023-06-21


In [8]:
assets.shape

(150, 14)

In [9]:
print("Total Assets:", len(assets))

print("\nAssets per Site:")
print(assets["Site_ID"].value_counts().sort_index())

print("\nDuplicate Asset_ID:")
print(assets["Asset_ID"].duplicated().sum())

print("\nDuplicate Serial_Number:")
print(assets["Serial_Number"].duplicated().sum())

print("\nMissing Values:")
print(assets.isnull().sum())

Total Assets: 150

Assets per Site:
Site_ID
SITE001    50
SITE002    50
SITE003    50
Name: count, dtype: int64

Duplicate Asset_ID:
0

Duplicate Serial_Number:
0

Missing Values:
Asset_ID               0
Site_ID                0
Asset_Name             0
Asset_Type             0
Manufacturer           0
Model                  0
Serial_Number          0
Install_Date           0
Criticality            0
Rated_Capacity         0
Capacity_Unit          0
Asset_Status           0
Expected_Life_Years    0
Warranty_End_Date      0
dtype: int64


In [10]:
invalid_sites = ~assets["Site_ID"].isin(sites["Site_ID"])

print("Invalid Site_ID references:", invalid_sites.sum())

Invalid Site_ID references: 0


In [ ]:
### Save the data

""" assets.to_csv("../data/assets.csv", index=False)

print("assets.csv created successfully.") """

assets.csv created successfully.


### Step 3 — Sensor Readings

### Objective
Generate 90 days of hourly operational sensor data for all 150 assets.

The readings will include:

- Temperature
- Vibration
- Pressure
- Power Consumption
- RPM
- Load %
- Health Score
- Operating State
- Anomaly Flag

Sensor values are generated using asset-type-specific operating ranges rather than purely random values.

### Expected Volume
150 assets × 90 days × 24 hours = 324,000 readings

### Output
`sensor_readings.csv`

In [12]:
sensor_profiles = {
    "Compressor": {
        "temperature": (60, 80),
        "vibration": (1.5, 4.0),
        "pressure": (6.0, 9.0),
        "power": (20, 55),
        "rpm": (1400, 1800)
    },

    "Pump": {
        "temperature": (45, 70),
        "vibration": (1.0, 3.5),
        "pressure": (3.0, 7.0),
        "power": (10, 40),
        "rpm": (1200, 1800)
    },

    "Motor": {
        "temperature": (50, 75),
        "vibration": (1.0, 4.0),
        "pressure": (0.0, 0.2),
        "power": (8, 75),
        "rpm": (1200, 1800)
    },

    "Fan": {
        "temperature": (35, 60),
        "vibration": (0.8, 3.0),
        "pressure": (0.5, 2.0),
        "power": (5, 30),
        "rpm": (800, 1500)
    },

    "Conveyor": {
        "temperature": (40, 65),
        "vibration": (1.0, 3.5),
        "pressure": (0.0, 0.2),
        "power": (5, 35),
        "rpm": (500, 1200)
    }
}

In [13]:
start_date = pd.Timestamp("2026-06-01 00:00:00")

timestamps = pd.date_range(
    start=start_date,
    periods=90 * 24,
    freq="h"
)

print("Start:", timestamps.min())
print("End:", timestamps.max())
print("Hours:", len(timestamps))

Start: 2026-06-01 00:00:00
End: 2026-08-29 23:00:00
Hours: 2160


In [14]:
## Generate sensor readings
np.random.seed(42)

sensor_records = []

reading_counter = 1

for _, asset in assets.iterrows():

    asset_id = asset["Asset_ID"]
    asset_type = asset["Asset_Type"]
    asset_status = asset["Asset_Status"]

    profile = sensor_profiles[asset_type]

    # Small asset-specific bias to avoid identical behaviour
    temp_bias = np.random.normal(0, 2)
    vibration_bias = np.random.normal(0, 0.25)
    power_bias = np.random.normal(0, 2)

    for timestamp in timestamps:

        # -----------------------------
        # Offline asset behaviour
        # -----------------------------
        if asset_status == "Offline":

            temperature = np.random.uniform(18, 28)
            vibration = 0
            pressure = 0
            power = 0
            rpm = 0
            load_pct = 0

            health_score = np.random.uniform(45, 75)

            operating_state = "Offline"
            anomaly_flag = False

        else:

            # -----------------------------
            # Normal operating load
            # -----------------------------
            load_pct = np.clip(
                np.random.normal(70, 15),
                20,
                100
            )

            temperature = np.random.normal(
                np.mean(profile["temperature"]) + temp_bias
                + (load_pct - 70) * 0.12,
                3
            )

            vibration = np.random.normal(
                np.mean(profile["vibration"]) + vibration_bias,
                0.4
            )

            pressure = np.random.normal(
                np.mean(profile["pressure"]),
                max(
                    (profile["pressure"][1] - profile["pressure"][0]) * 0.08,
                    0.02
                )
            )

            power = np.random.normal(
                np.mean(profile["power"]) + power_bias
                + (load_pct - 70) * 0.20,
                3
            )

            rpm = np.random.normal(
                np.mean(profile["rpm"]),
                60
            )

            # -----------------------------
            # Introduce realistic anomalies
            # Approx. 2% of operating readings
            # -----------------------------
            anomaly_flag = np.random.random() < 0.02

            if anomaly_flag:

                anomaly_type = np.random.choice([
                    "temperature",
                    "vibration",
                    "pressure",
                    "combined"
                ])

                if anomaly_type == "temperature":
                    temperature += np.random.uniform(12, 25)

                elif anomaly_type == "vibration":
                    vibration += np.random.uniform(2.5, 5.0)

                elif anomaly_type == "pressure":
                    if profile["pressure"][1] > 1:
                        pressure *= np.random.uniform(0.55, 0.75)

                elif anomaly_type == "combined":
                    temperature += np.random.uniform(10, 20)
                    vibration += np.random.uniform(2, 4)

            # -----------------------------
            # Safety clipping
            # -----------------------------
            temperature = max(0, temperature)
            vibration = max(0, vibration)
            pressure = max(0, pressure)
            power = max(0, power)
            rpm = max(0, rpm)

            # -----------------------------
            # Health Score
            # -----------------------------
            health_score = 100

            if temperature > profile["temperature"][1]:
                health_score -= min(
                    25,
                    (temperature - profile["temperature"][1]) * 2
                )

            if vibration > profile["vibration"][1]:
                health_score -= min(
                    30,
                    (vibration - profile["vibration"][1]) * 7
                )

            if (
                profile["pressure"][1] > 1
                and pressure < profile["pressure"][0]
            ):
                health_score -= 15

            if load_pct > 90:
                health_score -= 5

            health_score += np.random.normal(0, 2)

            health_score = np.clip(
                health_score,
                0,
                100
            )

            # -----------------------------
            # Operating State
            # -----------------------------
            if health_score < 50:
                operating_state = "Critical"

            elif health_score < 75:
                operating_state = "Warning"

            elif load_pct < 30:
                operating_state = "Idle"

            else:
                operating_state = "Running"

        sensor_records.append({
            "Reading_ID": f"RDG{reading_counter:07d}",
            "Asset_ID": asset_id,
            "Timestamp": timestamp,
            "Temperature_C": round(temperature, 2),
            "Vibration_mm_s": round(vibration, 2),
            "Pressure_Bar": round(pressure, 2),
            "Power_Consumption_kW": round(power, 2),
            "RPM": round(rpm),
            "Load_Pct": round(load_pct, 2),
            "Health_Score": round(health_score, 2),
            "Operating_State": operating_state,
            "Anomaly_Flag": anomaly_flag
        })

        reading_counter += 1


sensor_readings = pd.DataFrame(sensor_records)

sensor_readings.head()

,Reading_ID,Asset_ID,Timestamp,Temperature_C,Vibration_mm_s,Pressure_Bar,Power_Consumption_kW,RPM,Load_Pct,Health_Score,Operating_State,Anomaly_Flag
0,RDG0000001,AST0001,2026-06-01 00:00:00,50.53,1.77,1.44,25.67,1122,92.85,96.09,Running,False
1,RDG0000002,AST0001,2026-06-01 01:00:00,44.69,0.82,1.36,19.53,1059,61.43,98.18,Running,False
2,RDG0000003,AST0001,2026-06-01 02:00:00,50.35,1.78,1.26,10.28,1117,48.82,100.00,Running,False
3,RDG0000004,AST0001,2026-06-01 03:00:00,50.26,1.98,1.17,13.93,1140,61.00,100.00,Running,False
4,RDG0000005,AST0001,2026-06-01 04:00:00,46.92,1.08,1.09,15.72,1194,51.69,100.00,Running,False


In [15]:
print("Total Sensor Readings:", len(sensor_readings))

print("\nUnique Assets:")
print(sensor_readings["Asset_ID"].nunique())

print("\nDate Range:")
print(sensor_readings["Timestamp"].min())
print(sensor_readings["Timestamp"].max())

print("\nDuplicate Reading_ID:")
print(sensor_readings["Reading_ID"].duplicated().sum())

print("\nMissing Values:")
print(sensor_readings.isnull().sum())

Total Sensor Readings: 324000

Unique Assets:
150

Date Range:
2026-06-01 00:00:00
2026-08-29 23:00:00

Duplicate Reading_ID:
0

Missing Values:
Reading_ID              0
Asset_ID                0
Timestamp               0
Temperature_C           0
Vibration_mm_s          0
Pressure_Bar            0
Power_Consumption_kW    0
RPM                     0
Load_Pct                0
Health_Score            0
Operating_State         0
Anomaly_Flag            0
dtype: int64


In [16]:
sensor_readings["Anomaly_Flag"].value_counts(normalize=True) * 100

Anomaly_Flag
False    98.064815
True      1.935185
Name: proportion, dtype: float64

In [17]:
sensor_readings["Operating_State"].value_counts()

Operating_State
Running    311111
Offline     10800
Idle         1147
Warning       942
Name: count, dtype: int64

In [18]:
sensor_readings["Health_Score"].describe()

count    324000.000000
mean         97.257608
std           7.656591
min          45.010000
25%          97.830000
50%          99.610000
75%         100.000000
max         100.000000
Name: Health_Score, dtype: float64

In [19]:
invalid_assets = ~sensor_readings["Asset_ID"].isin(
    assets["Asset_ID"]
)

print(
    "Invalid Asset_ID references:",
    invalid_assets.sum()
)

Invalid Asset_ID references: 0


In [ ]:
### Save the data

""" sensor_readings.to_csv(
    "../data/sensor_readings.csv",
    index=False
)

print("sensor_readings.csv created successfully.") """

sensor_readings.csv created successfully.


### Step 4 — Failure Events

### Objective
Generate realistic equipment failure events using asset condition signals from the sensor dataset.

Failure probability will be influenced by:
- Low average health score
- High anomaly frequency
- Asset criticality
- Equipment type

### Output
`failures.csv`

In [21]:
# Asset-level condition summary
asset_condition = (
    sensor_readings
    .groupby("Asset_ID")
    .agg(
        Avg_Health_Score=("Health_Score", "mean"),
        Min_Health_Score=("Health_Score", "min"),
        Anomaly_Count=("Anomaly_Flag", "sum"),
        Reading_Count=("Reading_ID", "count")
    )
    .reset_index()
)

asset_condition["Anomaly_Rate"] = (
    asset_condition["Anomaly_Count"]
    / asset_condition["Reading_Count"]
)

asset_condition.head()

,Asset_ID,Avg_Health_Score,Min_Health_Score,Anomaly_Count,Reading_Count,Anomaly_Rate
0,AST0001,98.610819,62.14,33,2160,0.015278
1,AST0002,98.486463,56.86,48,2160,0.022222
2,AST0003,98.560222,57.58,38,2160,0.017593
3,AST0004,98.711389,70.72,33,2160,0.015278
4,AST0005,98.551356,62.57,56,2160,0.025926


In [22]:
failure_base = assets.merge(
    asset_condition,
    on="Asset_ID",
    how="left"
)

failure_base.head()

,Asset_ID,Site_ID,Asset_Name,Asset_Type,Manufacturer,Model,Serial_Number,Install_Date,Criticality,Rated_Capacity,Capacity_Unit,Asset_Status,Expected_Life_Years,Warranty_End_Date,Avg_Health_Score,Min_Health_Score,Anomaly_Count,Reading_Count,Anomaly_Rate
0,AST0001,SITE001,Fan 001,Fan,ABB,AXIAL-500,SN-000001,2018-02-04,Low,11687.49,m3/h,Active,15,2023-02-04,98.610819,62.14,33,2160,0.015278
1,AST0002,SITE001,Motor 002,Motor,Siemens,M3BP,SN-000002,2024-06-06,Medium,11.65,kW,Active,12,2029-06-06,98.486463,56.86,48,2160,0.022222
2,AST0003,SITE001,Fan 003,Fan,FläktGroup,CENT-750,SN-000003,2021-08-30,High,9563.63,m3/h,Active,12,2026-08-30,98.560222,57.58,38,2160,0.017593
3,AST0004,SITE001,Conveyor 004,Conveyor,Dorner,MOVIMOT,SN-000004,2024-12-11,Medium,593.33,kg/h,Active,20,2029-12-11,98.711389,70.72,33,2160,0.015278
4,AST0005,SITE001,Motor 005,Motor,ABB,SIMOTICS GP,SN-000005,2018-06-21,Medium,58.60,kW,Active,10,2023-06-21,98.551356,62.57,56,2160,0.025926


In [23]:
np.random.seed(42)

failure_records = []
failure_counter = 1

failure_types = {
    "Compressor": [
        ("Overheating", "Bearing"),
        ("Pressure Loss", "Seal"),
        ("Bearing Failure", "Bearing")
    ],
    "Pump": [
        ("Pressure Loss", "Seal"),
        ("Excessive Vibration", "Impeller"),
        ("Motor Failure", "Motor")
    ],
    "Motor": [
        ("Overheating", "Winding"),
        ("Bearing Failure", "Bearing"),
        ("Electrical Fault", "Electrical System")
    ],
    "Fan": [
        ("Excessive Vibration", "Fan Blade"),
        ("Motor Failure", "Motor"),
        ("Bearing Failure", "Bearing")
    ],
    "Conveyor": [
        ("Mechanical Wear", "Belt"),
        ("Motor Failure", "Drive Motor"),
        ("Bearing Failure", "Roller Bearing")
    ]
}

for _, asset in failure_base.iterrows():

    # Base failure probability
    failure_probability = 0.10

    # Poor health increases risk
    if asset["Avg_Health_Score"] < 90:
        failure_probability += 0.10

    if asset["Avg_Health_Score"] < 80:
        failure_probability += 0.15

    # Higher anomaly rate increases risk
    if asset["Anomaly_Rate"] > 0.02:
        failure_probability += 0.10

    # Critical equipment slightly more likely
    if asset["Criticality"] == "High":
        failure_probability += 0.05

    # Cap probability
    failure_probability = min(
        failure_probability,
        0.55
    )

    # Decide whether asset has failure
    if np.random.random() < failure_probability:

        # 1–3 failures
        num_failures = np.random.choice(
            [1, 2, 3],
            p=[0.70, 0.25, 0.05]
        )

        asset_readings = sensor_readings[
            sensor_readings["Asset_ID"]
            == asset["Asset_ID"]
        ]

        # Prefer anomalous timestamps
        anomaly_rows = asset_readings[
            asset_readings["Anomaly_Flag"] == True
        ]

        for _ in range(num_failures):

            if len(anomaly_rows) > 0:
                selected_row = anomaly_rows.sample(
                    1,
                    random_state=None
                ).iloc[0]

            else:
                selected_row = asset_readings.sample(
                    1,
                    random_state=None
                ).iloc[0]

            failure_datetime = selected_row["Timestamp"]

            failure_type, component = (
                failure_types[
                    asset["Asset_Type"]
                ][
                    np.random.randint(
                        len(
                            failure_types[
                                asset["Asset_Type"]
                            ]
                        )
                    )
                ]
            )

            severity = np.random.choice(
                ["Low", "Medium", "High", "Critical"],
                p=[0.15, 0.40, 0.30, 0.15]
            )

            downtime_map = {
                "Low": (0.5, 2),
                "Medium": (2, 5),
                "High": (5, 12),
                "Critical": (12, 36)
            }

            downtime_hours = round(
                np.random.uniform(
                    *downtime_map[severity]
                ),
                2
            )

            detection_method = np.random.choice(
                [
                    "Sensor Alert",
                    "Predictive Alert",
                    "Operator Inspection",
                    "Scheduled Inspection"
                ],
                p=[0.40, 0.25, 0.25, 0.10]
            )

            root_cause = np.random.choice([
                "Lubrication Failure",
                "Misalignment",
                "Component Wear",
                "Overload",
                "Electrical Degradation",
                "Seal Degradation"
            ])

            production_impact = (
                "Major"
                if severity == "Critical"
                else "Moderate"
                if severity == "High"
                else "Minor"
            )

            failure_records.append({
                "Failure_ID":
                    f"FLR{failure_counter:04d}",

                "Asset_ID":
                    asset["Asset_ID"],

                "Failure_DateTime":
                    failure_datetime,

                "Failure_Type":
                    failure_type,

                "Failure_Component":
                    component,

                "Severity":
                    severity,

                "Downtime_Hours":
                    downtime_hours,

                "Detection_Method":
                    detection_method,

                "Root_Cause":
                    root_cause,

                "Production_Impact":
                    production_impact,

                "Failure_Status":
                    "Resolved"
            })

            failure_counter += 1


failures = pd.DataFrame(failure_records)

failures.head()

,Failure_ID,Asset_ID,Failure_DateTime,Failure_Type,Failure_Component,Severity,Downtime_Hours,Detection_Method,Root_Cause,Production_Impact,Failure_Status
0,FLR0001,AST0005,2026-08-29 03:00:00,Electrical Fault,Electrical System,Medium,3.17,Sensor Alert,Overload,Minor,Resolved
1,FLR0002,AST0009,2026-07-03 03:00:00,Motor Failure,Motor,Medium,2.98,Operator Inspection,Seal Degradation,Minor,Resolved
2,FLR0003,AST0009,2026-08-13 04:00:00,Motor Failure,Motor,Medium,2.37,Sensor Alert,Seal Degradation,Minor,Resolved
3,FLR0004,AST0009,2026-07-21 10:00:00,Excessive Vibration,Impeller,Critical,17.81,Operator Inspection,Component Wear,Major,Resolved
4,FLR0005,AST0018,2026-08-11 23:00:00,Pressure Loss,Seal,High,9.06,Sensor Alert,Lubrication Failure,Moderate,Resolved


In [24]:
print("Total Failures:", len(failures))

print("\nUnique Assets with Failures:")
print(failures["Asset_ID"].nunique())

print("\nFailures by Severity:")
print(failures["Severity"].value_counts())

print("\nFailures by Type:")
print(failures["Failure_Type"].value_counts())

print("\nDuplicate Failure_ID:")
print(failures["Failure_ID"].duplicated().sum())

print("\nMissing Values:")
print(failures.isnull().sum())

Total Failures: 40

Unique Assets with Failures:
29

Failures by Severity:
Severity
Medium      16
Critical     8
High         8
Low          8
Name: count, dtype: int64

Failures by Type:
Failure_Type
Bearing Failure        9
Excessive Vibration    8
Motor Failure          7
Mechanical Wear        6
Pressure Loss          4
Overheating            4
Electrical Fault       2
Name: count, dtype: int64

Duplicate Failure_ID:
0

Missing Values:
Failure_ID           0
Asset_ID             0
Failure_DateTime     0
Failure_Type         0
Failure_Component    0
Severity             0
Downtime_Hours       0
Detection_Method     0
Root_Cause           0
Production_Impact    0
Failure_Status       0
dtype: int64


In [25]:
invalid_assets = ~failures[
    "Asset_ID"
].isin(
    assets["Asset_ID"]
)

print(
    "Invalid Asset_ID references:",
    invalid_assets.sum()
)

Invalid Asset_ID references: 0


In [ ]:
## Save the data

""" failures.to_csv(
    "../data/failures.csv",
    index=False
)

print("failures.csv created successfully.") """

failures.csv created successfully.


### Step 5 — Maintenance Events

### Objective
Generate realistic maintenance history for all assets.

Maintenance records will include:
- Preventive maintenance
- Inspections
- Corrective maintenance
- Emergency maintenance
- Failure-linked repairs

### Output
`maintenance.csv`

In [27]:
np.random.seed(42)

maintenance_records = []
maintenance_counter = 1

components_by_asset = {
    "Compressor": ["Bearing", "Seal", "Lubrication System", "Motor"],
    "Pump": ["Seal", "Impeller", "Motor", "Bearing"],
    "Motor": ["Bearing", "Winding", "Electrical System"],
    "Fan": ["Fan Blade", "Bearing", "Motor"],
    "Conveyor": ["Belt", "Drive Motor", "Roller Bearing"]
}

# --------------------------------------------------
# 1. Planned maintenance for all assets
# --------------------------------------------------

for _, asset in assets.iterrows():

    num_planned_events = np.random.choice(
        [1, 2, 3],
        p=[0.35, 0.50, 0.15]
    )

    for _ in range(num_planned_events):

        maintenance_type = np.random.choice(
            ["Preventive", "Inspection"],
            p=[0.75, 0.25]
        )

        maintenance_date = (
            start_date
            + pd.to_timedelta(
                np.random.randint(0, 90 * 24),
                unit="h"
            )
        )

        component = np.random.choice(
            components_by_asset[asset["Asset_Type"]]
        )

        duration = round(
            np.random.uniform(0.5, 4.0),
            2
        )

        parts_replaced = (
            component
            if maintenance_type == "Preventive"
            and np.random.random() < 0.25
            else "None"
        )

        maintenance_records.append({
            "Maintenance_ID": f"MNT{maintenance_counter:05d}",
            "Asset_ID": asset["Asset_ID"],
            "Failure_ID": None,
            "Maintenance_Date": maintenance_date,
            "Maintenance_Type": maintenance_type,
            "Maintenance_Reason": (
                "Scheduled preventive maintenance"
                if maintenance_type == "Preventive"
                else "Routine equipment inspection"
            ),
            "Component": component,
            "Technician_ID": f"TECH{np.random.randint(1, 16):03d}",
            "Duration_Hours": duration,
            "Planned_Flag": True,
            "Maintenance_Status": "Completed",
            "Parts_Replaced": parts_replaced,
            "Notes": "Planned maintenance completed"
        })

        maintenance_counter += 1


# --------------------------------------------------
# 2. Failure-driven maintenance
# --------------------------------------------------

for _, failure in failures.iterrows():

    severity = failure["Severity"]

    if severity == "Critical":
        maintenance_type = "Emergency"
    else:
        maintenance_type = "Corrective"

    maintenance_date = (
        pd.to_datetime(failure["Failure_DateTime"])
        + pd.to_timedelta(
            np.random.uniform(0.25, 4),
            unit="h"
        )
    )

    duration_multiplier = {
        "Low": 0.8,
        "Medium": 1.0,
        "High": 1.15,
        "Critical": 1.25
    }

    duration = round(
        max(
            0.5,
            failure["Downtime_Hours"]
            * duration_multiplier[severity]
            * np.random.uniform(0.65, 0.95)
        ),
        2
    )

    part_replaced = (
        failure["Failure_Component"]
        if np.random.random() < 0.70
        else "None"
    )

    maintenance_records.append({
        "Maintenance_ID": f"MNT{maintenance_counter:05d}",
        "Asset_ID": failure["Asset_ID"],
        "Failure_ID": failure["Failure_ID"],
        "Maintenance_Date": maintenance_date,
        "Maintenance_Type": maintenance_type,
        "Maintenance_Reason": failure["Failure_Type"],
        "Component": failure["Failure_Component"],
        "Technician_ID": f"TECH{np.random.randint(1, 16):03d}",
        "Duration_Hours": duration,
        "Planned_Flag": False,
        "Maintenance_Status": "Completed",
        "Parts_Replaced": part_replaced,
        "Notes": f"Maintenance performed for {failure['Failure_ID']}"
    })

    maintenance_counter += 1


maintenance = pd.DataFrame(maintenance_records)

maintenance.head()

,Maintenance_ID,Asset_ID,Failure_ID,Maintenance_Date,Maintenance_Type,Maintenance_Reason,Component,Technician_ID,Duration_Hours,Planned_Flag,Maintenance_Status,Parts_Replaced,Notes
0,MNT00001,AST0001,NaN,2026-07-18 02:00:00,Inspection,Routine equipment inspection,Fan Blade,TECH010,2.59,True,Completed,None,Planned maintenance completed
1,MNT00002,AST0001,NaN,2026-06-14 18:00:00,Preventive,Scheduled preventive maintenance,Motor,TECH008,3.53,True,Completed,None,Planned maintenance completed
2,MNT00003,AST0002,NaN,2026-08-03 03:00:00,Preventive,Scheduled preventive maintenance,Winding,TECH005,1.24,True,Completed,Winding,Planned maintenance completed
3,MNT00004,AST0002,NaN,2026-06-01 21:00:00,Preventive,Scheduled preventive maintenance,Bearing,TECH011,2.01,True,Completed,None,Planned maintenance completed
4,MNT00005,AST0003,NaN,2026-07-11 15:00:00,Preventive,Scheduled preventive maintenance,Motor,TECH003,2.10,True,Completed,None,Planned maintenance completed


In [28]:
print("Total Maintenance Records:", len(maintenance))

print("\nMaintenance by Type:")
print(maintenance["Maintenance_Type"].value_counts())

print("\nPlanned vs Unplanned:")
print(maintenance["Planned_Flag"].value_counts())

print("\nDuplicate Maintenance_ID:")
print(maintenance["Maintenance_ID"].duplicated().sum())

print("\nMissing Values:")
print(maintenance.isnull().sum())

Total Maintenance Records: 310

Maintenance by Type:
Maintenance_Type
Preventive    197
Inspection     73
Corrective     32
Emergency       8
Name: count, dtype: int64

Planned vs Unplanned:
Planned_Flag
True     270
False     40
Name: count, dtype: int64

Duplicate Maintenance_ID:
0

Missing Values:
Maintenance_ID          0
Asset_ID                0
Failure_ID            270
Maintenance_Date        0
Maintenance_Type        0
Maintenance_Reason      0
Component               0
Technician_ID           0
Duration_Hours          0
Planned_Flag            0
Maintenance_Status      0
Parts_Replaced          0
Notes                   0
dtype: int64


In [29]:
invalid_assets = ~maintenance["Asset_ID"].isin(
    assets["Asset_ID"]
)

print(
    "Invalid Asset_ID references:",
    invalid_assets.sum()
)

Invalid Asset_ID references: 0


In [30]:
linked_failure_ids = maintenance[
    maintenance["Failure_ID"].notna()
]["Failure_ID"]

invalid_failures = ~linked_failure_ids.isin(
    failures["Failure_ID"]
)

print(
    "Invalid Failure_ID references:",
    invalid_failures.sum()
)

Invalid Failure_ID references: 0


In [31]:
failure_maintenance_check = (
    failures["Failure_ID"]
    .isin(
        maintenance["Failure_ID"].dropna()
    )
)

print(
    "Failures linked to maintenance:",
    failure_maintenance_check.sum(),
    "/",
    len(failures)
)

Failures linked to maintenance: 40 / 40


In [ ]:
## Save the data

""" maintenance.to_csv(
    "../data/maintenance.csv",
    index=False
)

print("maintenance.csv created successfully.") """

maintenance.csv created successfully.


### Step 6 — Work Orders

### Objective
Generate operational work orders linked to maintenance activity and failure events.

The dataset will capture:
- Work order type
- Priority
- Assigned team
- Estimated vs actual hours
- Completion status
- SLA targets
- SLA breach flag

### Output
`work_orders.csv`

In [33]:
np.random.seed(42)

work_order_records = []
work_order_counter = 1

for _, row in maintenance.iterrows():

    maintenance_type = row["Maintenance_Type"]

    # -----------------------------
    # Work Order Type
    # -----------------------------
    work_order_type_map = {
        "Preventive": "Preventive Maintenance",
        "Inspection": "Inspection",
        "Corrective": "Corrective Repair",
        "Emergency": "Emergency Repair",
        "Predictive": "Predictive Intervention"
    }

    work_order_type = work_order_type_map.get(
        maintenance_type,
        "Maintenance Task"
    )

    # -----------------------------
    # Priority
    # -----------------------------
    if maintenance_type == "Emergency":
        priority = "Critical"

    elif maintenance_type == "Corrective":
        priority = np.random.choice(
            ["Medium", "High"],
            p=[0.35, 0.65]
        )

    elif maintenance_type == "Predictive":
        priority = "High"

    else:
        priority = np.random.choice(
            ["Low", "Medium"],
            p=[0.35, 0.65]
        )

    # -----------------------------
    # Dates
    # -----------------------------
    maintenance_date = pd.to_datetime(
        row["Maintenance_Date"]
    )

    created_datetime = (
        maintenance_date
        - pd.to_timedelta(
            np.random.uniform(
                1 if row["Planned_Flag"] else 0.25,
                72 if row["Planned_Flag"] else 6
            ),
            unit="h"
        )
    )

    scheduled_datetime = (
        maintenance_date
        - pd.to_timedelta(
            np.random.uniform(0, 2),
            unit="h"
        )
    )

    # -----------------------------
    # Estimated & Actual Hours
    # -----------------------------
    actual_hours = float(row["Duration_Hours"])

    estimated_hours = round(
        max(
            0.5,
            actual_hours * np.random.uniform(0.80, 1.20)
        ),
        2
    )

    completed_datetime = (
        maintenance_date
        + pd.to_timedelta(
            actual_hours,
            unit="h"
        )
    )

    # -----------------------------
    # SLA
    # -----------------------------
    sla_map = {
        "Low": 72,
        "Medium": 48,
        "High": 24,
        "Critical": 8
    }

    sla_target_hours = sla_map[priority]

    resolution_hours = (
        completed_datetime - created_datetime
    ).total_seconds() / 3600

    sla_breached = (
        resolution_hours > sla_target_hours
    )

    # -----------------------------
    # Assigned Team
    # -----------------------------
    assigned_team = np.random.choice([
        "Mechanical Team",
        "Electrical Team",
        "Reliability Team",
        "Maintenance Team A",
        "Maintenance Team B"
    ])

    work_order_records.append({
        "WorkOrder_ID":
            f"WO{work_order_counter:05d}",

        "Asset_ID":
            row["Asset_ID"],

        "Maintenance_ID":
            row["Maintenance_ID"],

        "Failure_ID":
            row["Failure_ID"],

        "Created_DateTime":
            created_datetime,

        "Scheduled_DateTime":
            scheduled_datetime,

        "Completed_DateTime":
            completed_datetime,

        "WorkOrder_Type":
            work_order_type,

        "Priority":
            priority,

        "Assigned_Team":
            assigned_team,

        "WorkOrder_Status":
            "Completed",

        "Estimated_Hours":
            estimated_hours,

        "Actual_Hours":
            actual_hours,

        "Description":
            row["Maintenance_Reason"],

        "SLA_Target_Hours":
            sla_target_hours,

        "SLA_Breached_Flag":
            sla_breached
    })

    work_order_counter += 1


work_orders = pd.DataFrame(
    work_order_records
)

work_orders.head()

,WorkOrder_ID,Asset_ID,Maintenance_ID,Failure_ID,Created_DateTime,Scheduled_DateTime,Completed_DateTime,WorkOrder_Type,Priority,Assigned_Team,WorkOrder_Status,Estimated_Hours,Actual_Hours,Description,SLA_Target_Hours,SLA_Breached_Flag
0,WO00001,AST0001,MNT00001,NaN,2026-07-15 05:29:57.423281626,2026-07-18 00:32:09.643618958,2026-07-18 04:35:24,Inspection,Medium,Electrical Team,Completed,2.69,2.59,Routine equipment inspection,48,True
1,WO00002,AST0001,MNT00002,NaN,2026-06-14 12:52:33.828729809,2026-06-14 16:16:03.531750420,2026-06-14 21:31:48,Preventive Maintenance,Low,Reliability Team,Completed,3.67,3.53,Scheduled preventive maintenance,72,False
2,WO00003,AST0002,MNT00003,NaN,2026-07-31 05:08:11.041787393,2026-08-03 01:20:06.412986237,2026-08-03 04:14:24,Preventive Maintenance,Low,Maintenance Team A,Completed,1.10,1.24,Scheduled preventive maintenance,72,False
3,WO00004,AST0002,MNT00004,NaN,2026-05-31 22:23:55.682699543,2026-06-01 19:57:01.753692250,2026-06-01 23:00:36,Preventive Maintenance,Low,Mechanical Team,Completed,1.96,2.01,Scheduled preventive maintenance,72,False
4,WO00005,AST0003,MNT00005,NaN,2026-07-10 09:36:35.535629582,2026-07-11 14:54:24.007224863,2026-07-11 17:06:00,Preventive Maintenance,Medium,Reliability Team,Completed,2.50,2.10,Scheduled preventive maintenance,48,False


In [34]:
print("Total Work Orders:", len(work_orders))

print("\nWork Orders by Type:")
print(work_orders["WorkOrder_Type"].value_counts())

print("\nPriority Distribution:")
print(work_orders["Priority"].value_counts())

print("\nSLA Breach Distribution:")
print(work_orders["SLA_Breached_Flag"].value_counts())

print("\nDuplicate WorkOrder_ID:")
print(
    work_orders["WorkOrder_ID"]
    .duplicated()
    .sum()
)

print("\nMissing Values:")
print(work_orders.isnull().sum())

Total Work Orders: 310

Work Orders by Type:
WorkOrder_Type
Preventive Maintenance    197
Inspection                 73
Corrective Repair          32
Emergency Repair            8
Name: count, dtype: int64

Priority Distribution:
Priority
Medium      171
Low         106
High         25
Critical      8
Name: count, dtype: int64

SLA Breach Distribution:
SLA_Breached_Flag
False    228
True      82
Name: count, dtype: int64

Duplicate WorkOrder_ID:
0

Missing Values:
WorkOrder_ID            0
Asset_ID                0
Maintenance_ID          0
Failure_ID            270
Created_DateTime        0
Scheduled_DateTime      0
Completed_DateTime      0
WorkOrder_Type          0
Priority                0
Assigned_Team           0
WorkOrder_Status        0
Estimated_Hours         0
Actual_Hours            0
Description             0
SLA_Target_Hours        0
SLA_Breached_Flag       0
dtype: int64


In [35]:
print(
    "Invalid Asset_ID:",
    (
        ~work_orders["Asset_ID"]
        .isin(assets["Asset_ID"])
    ).sum()
)

print(
    "Invalid Maintenance_ID:",
    (
        ~work_orders["Maintenance_ID"]
        .isin(maintenance["Maintenance_ID"])
    ).sum()
)

Invalid Asset_ID: 0
Invalid Maintenance_ID: 0


In [36]:
wo_failure_ids = work_orders[
    work_orders["Failure_ID"].notna()
]["Failure_ID"]

print(
    "Invalid Failure_ID:",
    (
        ~wo_failure_ids.isin(
            failures["Failure_ID"]
        )
    ).sum()
)

Invalid Failure_ID: 0


In [37]:
print(
    "Unique Maintenance IDs in Work Orders:",
    work_orders["Maintenance_ID"].nunique()
)

print(
    "Maintenance Records:",
    len(maintenance)
)

Unique Maintenance IDs in Work Orders: 310
Maintenance Records: 310


In [ ]:
## Save the data
""" 
work_orders.to_csv(
    "../data/work_orders.csv",
    index=False
)

print("work_orders.csv created successfully.") """

work_orders.csv created successfully.


### Step 7 — Cost Records

### Objective
Generate cost records for each maintenance work order.

The dataset will capture:
- Labour cost
- Parts cost
- Contractor cost
- Downtime cost
- Other cost
- Total cost

Costs will vary by maintenance type, priority, duration, and failure severity.

### Output
`costs.csv`

In [39]:
np.random.seed(42)

cost_records = []
cost_counter = 1

# Labour rates per hour
labour_rate_map = {
    "Mechanical Team": 55,
    "Electrical Team": 60,
    "Reliability Team": 65,
    "Maintenance Team A": 50,
    "Maintenance Team B": 50
}

for _, wo in work_orders.iterrows():

    actual_hours = float(wo["Actual_Hours"])

    labour_rate = labour_rate_map[
        wo["Assigned_Team"]
    ]

    # -----------------------------
    # Labour Cost
    # -----------------------------
    labour_cost = round(
        actual_hours * labour_rate,
        2
    )

    # -----------------------------
    # Parts Cost
    # -----------------------------
    if wo["WorkOrder_Type"] == "Inspection":
        parts_cost = np.random.uniform(0, 50)

    elif wo["WorkOrder_Type"] == "Preventive Maintenance":
        parts_cost = np.random.uniform(20, 300)

    elif wo["WorkOrder_Type"] == "Corrective Repair":
        parts_cost = np.random.uniform(150, 1200)

    elif wo["WorkOrder_Type"] == "Emergency Repair":
        parts_cost = np.random.uniform(500, 2500)

    else:
        parts_cost = np.random.uniform(50, 500)

    parts_cost = round(parts_cost, 2)

    # -----------------------------
    # Contractor Cost
    # -----------------------------
    contractor_cost = 0

    if np.random.random() < 0.12:
        contractor_cost = round(
            np.random.uniform(250, 1200),
            2
        )

    # -----------------------------
    # Downtime Cost
    # -----------------------------
    downtime_cost = 0

    if pd.notna(wo["Failure_ID"]):

        failure_row = failures.loc[
            failures["Failure_ID"]
            == wo["Failure_ID"]
        ]

        if not failure_row.empty:

            downtime_hours = float(
                failure_row.iloc[0]["Downtime_Hours"]
            )

            severity = failure_row.iloc[0]["Severity"]

            downtime_rate_map = {
                "Low": 150,
                "Medium": 300,
                "High": 600,
                "Critical": 1000
            }

            downtime_cost = round(
                downtime_hours
                * downtime_rate_map[severity],
                2
            )

    # -----------------------------
    # Other Cost
    # -----------------------------
    other_cost = round(
        np.random.uniform(0, 100),
        2
    )

    # -----------------------------
    # Total Cost
    # -----------------------------
    total_cost = round(
        labour_cost
        + parts_cost
        + contractor_cost
        + downtime_cost
        + other_cost,
        2
    )

    # -----------------------------
    # Cost Type
    # -----------------------------
    cost_type_map = {
        "Preventive Maintenance":
            "Preventive Maintenance",

        "Corrective Repair":
            "Corrective Maintenance",

        "Emergency Repair":
            "Emergency Repair",

        "Inspection":
            "Inspection",

        "Predictive Intervention":
            "Predictive Maintenance"
    }

    cost_type = cost_type_map.get(
        wo["WorkOrder_Type"],
        "Maintenance"
    )

    cost_records.append({
        "Cost_ID":
            f"CST{cost_counter:05d}",

        "Asset_ID":
            wo["Asset_ID"],

        "WorkOrder_ID":
            wo["WorkOrder_ID"],

        "Maintenance_ID":
            wo["Maintenance_ID"],

        "Failure_ID":
            wo["Failure_ID"],

        "Cost_Date":
            pd.to_datetime(
                wo["Completed_DateTime"]
            ).date(),

        "Cost_Type":
            cost_type,

        "Labour_Cost":
            labour_cost,

        "Parts_Cost":
            parts_cost,

        "Contractor_Cost":
            contractor_cost,

        "Downtime_Cost":
            downtime_cost,

        "Other_Cost":
            other_cost,

        "Total_Cost":
            total_cost,

        "Currency":
            "GBP"
    })

    cost_counter += 1


costs = pd.DataFrame(cost_records)

costs.head()

,Cost_ID,Asset_ID,WorkOrder_ID,Maintenance_ID,Failure_ID,Cost_Date,Cost_Type,Labour_Cost,Parts_Cost,Contractor_Cost,Downtime_Cost,Other_Cost,Total_Cost,Currency
0,CST00001,AST0001,WO00001,MNT00001,NaN,2026-07-18,Inspection,155.40,18.73,0.00,0.0,73.20,247.33,GBP
1,CST00002,AST0001,WO00002,MNT00002,NaN,2026-06-14,Preventive Maintenance,229.45,187.62,0.00,0.0,15.60,432.67,GBP
2,CST00003,AST0002,WO00003,MNT00003,NaN,2026-08-03,Preventive Maintenance,62.00,36.26,0.00,0.0,60.11,158.37,GBP
3,CST00004,AST0002,WO00004,MNT00004,NaN,2026-06-01,Preventive Maintenance,110.55,218.26,1171.41,0.0,83.24,1583.46,GBP
4,CST00005,AST0003,WO00005,MNT00005,NaN,2026-07-11,Preventive Maintenance,136.50,79.45,0.00,0.0,18.34,234.29,GBP


In [40]:
print("Total Cost Records:", len(costs))

print("\nCost by Type:")
print(costs["Cost_Type"].value_counts())

print("\nDuplicate Cost_ID:")
print(
    costs["Cost_ID"]
    .duplicated()
    .sum()
)

print("\nMissing Values:")
print(costs.isnull().sum())

print("\nTotal Cost Summary:")
print(
    costs["Total_Cost"]
    .describe()
)

Total Cost Records: 310

Cost by Type:
Cost_Type
Preventive Maintenance    197
Inspection                 73
Corrective Maintenance     32
Emergency Repair            8
Name: count, dtype: int64

Duplicate Cost_ID:
0

Missing Values:
Cost_ID              0
Asset_ID             0
WorkOrder_ID         0
Maintenance_ID       0
Failure_ID         270
Cost_Date            0
Cost_Type            0
Labour_Cost          0
Parts_Cost           0
Contractor_Cost      0
Downtime_Cost        0
Other_Cost           0
Total_Cost           0
Currency             0
dtype: int64

Total Cost Summary:
count      310.000000
mean      1305.219935
std       4323.092526
min         46.290000
25%        249.717500
50%        381.910000
75%        665.780000
max      38365.930000
Name: Total_Cost, dtype: float64


In [41]:
print(
    "Invalid Asset_ID:",
    (
        ~costs["Asset_ID"]
        .isin(assets["Asset_ID"])
    ).sum()
)

print(
    "Invalid WorkOrder_ID:",
    (
        ~costs["WorkOrder_ID"]
        .isin(work_orders["WorkOrder_ID"])
    ).sum()
)

print(
    "Invalid Maintenance_ID:",
    (
        ~costs["Maintenance_ID"]
        .isin(maintenance["Maintenance_ID"])
    ).sum()
)

Invalid Asset_ID: 0
Invalid WorkOrder_ID: 0
Invalid Maintenance_ID: 0


In [42]:
calculated_total = (
    costs["Labour_Cost"]
    + costs["Parts_Cost"]
    + costs["Contractor_Cost"]
    + costs["Downtime_Cost"]
    + costs["Other_Cost"]
).round(2)

print(
    "Incorrect Total_Cost rows:",
    (
        calculated_total
        != costs["Total_Cost"]
    ).sum()
)

Incorrect Total_Cost rows: 0


In [ ]:
## Save the data

""" costs.to_csv(
    "../data/costs.csv",
    index=False
)

print("costs.csv created successfully.") """

costs.csv created successfully.


### Step 8 — Data Quality & Relationship Validation

### Objective
Validate all seven datasets before loading them into Microsoft Fabric.

Checks include:
- Row counts
- Primary key uniqueness
- Missing values
- Foreign-key integrity
- Valid business-rule values
- Date consistency
- Cost calculation accuracy
- Cross-table relationship integrity

### Datasets
- sites
- assets
- sensor_readings
- failures
- maintenance
- work_orders
- costs

In [45]:
datasets = {
    "sites": sites,
    "assets": assets,
    "sensor_readings": sensor_readings,
    "failures": failures,
    "maintenance": maintenance,
    "work_orders": work_orders,
    "costs": costs
}

summary = []

for name, df in datasets.items():
    summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing_Values": int(df.isnull().sum().sum()),
        "Duplicate_Rows": int(df.duplicated().sum())
    })

data_quality_summary = pd.DataFrame(summary)

data_quality_summary

,Dataset,Rows,Columns,Missing_Values,Duplicate_Rows
0,sites,3,7,0,0
1,assets,150,14,0,0
2,sensor_readings,324000,12,0,0
3,failures,40,11,0,0
4,maintenance,310,13,270,0
5,work_orders,310,16,270,0
6,costs,310,14,270,0


In [ ]:
##Primary-key validation

primary_keys = {
    "sites": "Site_ID",
    "assets": "Asset_ID",
    "sensor_readings": "Reading_ID",
    "failures": "Failure_ID",
    "maintenance": "Maintenance_ID",
    "work_orders": "WorkOrder_ID",
    "costs": "Cost_ID"
}

pk_results = []

for name, pk in primary_keys.items():

    df = datasets[name]

    pk_results.append({
        "Dataset": name,
        "Primary_Key": pk,
        "Null_PK": int(df[pk].isnull().sum()),
        "Duplicate_PK": int(df[pk].duplicated().sum()),
        "Unique_PK": int(df[pk].nunique())
    })

pk_validation = pd.DataFrame(pk_results)

pk_validation

,Dataset,Primary_Key,Null_PK,Duplicate_PK,Unique_PK
0,sites,Site_ID,0,0,3
1,assets,Asset_ID,0,0,150
2,sensor_readings,Reading_ID,0,0,324000
3,failures,Failure_ID,0,0,40
4,maintenance,Maintenance_ID,0,0,310
5,work_orders,WorkOrder_ID,0,0,310
6,costs,Cost_ID,0,0,310


In [ ]:
## Foreign-key integrity
fk_checks = {
    "assets.Site_ID":
        (~assets["Site_ID"].isin(sites["Site_ID"])).sum(),

    "sensor_readings.Asset_ID":
        (~sensor_readings["Asset_ID"].isin(assets["Asset_ID"])).sum(),

    "failures.Asset_ID":
        (~failures["Asset_ID"].isin(assets["Asset_ID"])).sum(),

    "maintenance.Asset_ID":
        (~maintenance["Asset_ID"].isin(assets["Asset_ID"])).sum(),

    "work_orders.Asset_ID":
        (~work_orders["Asset_ID"].isin(assets["Asset_ID"])).sum(),

    "costs.Asset_ID":
        (~costs["Asset_ID"].isin(assets["Asset_ID"])).sum()
}

pd.Series(
    fk_checks,
    name="Invalid_References"
)

assets.Site_ID              0
sensor_readings.Asset_ID    0
failures.Asset_ID           0
maintenance.Asset_ID        0
work_orders.Asset_ID        0
costs.Asset_ID              0
Name: Invalid_References, dtype: int64

In [48]:
optional_fk_checks = {
    "maintenance.Failure_ID":
        (
            ~maintenance.loc[
                maintenance["Failure_ID"].notna(),
                "Failure_ID"
            ].isin(failures["Failure_ID"])
        ).sum(),

    "work_orders.Maintenance_ID":
        (
            ~work_orders["Maintenance_ID"]
            .isin(maintenance["Maintenance_ID"])
        ).sum(),

    "work_orders.Failure_ID":
        (
            ~work_orders.loc[
                work_orders["Failure_ID"].notna(),
                "Failure_ID"
            ].isin(failures["Failure_ID"])
        ).sum(),

    "costs.WorkOrder_ID":
        (
            ~costs["WorkOrder_ID"]
            .isin(work_orders["WorkOrder_ID"])
        ).sum(),

    "costs.Maintenance_ID":
        (
            ~costs["Maintenance_ID"]
            .isin(maintenance["Maintenance_ID"])
        ).sum(),

    "costs.Failure_ID":
        (
            ~costs.loc[
                costs["Failure_ID"].notna(),
                "Failure_ID"
            ].isin(failures["Failure_ID"])
        ).sum()
}

pd.Series(
    optional_fk_checks,
    name="Invalid_References"
)

maintenance.Failure_ID        0
work_orders.Maintenance_ID    0
work_orders.Failure_ID        0
costs.WorkOrder_ID            0
costs.Maintenance_ID          0
costs.Failure_ID              0
Name: Invalid_References, dtype: int64

In [49]:
## Business-rule validation
business_rule_checks = {
    "Invalid Criticality":
        (~assets["Criticality"].isin(
            ["High", "Medium", "Low"]
        )).sum(),

    "Invalid Asset Status":
        (~assets["Asset_Status"].isin(
            ["Active", "Maintenance", "Offline"]
        )).sum(),

    "Invalid Load Percentage":
        (
            (sensor_readings["Load_Pct"] < 0)
            | (sensor_readings["Load_Pct"] > 100)
        ).sum(),

    "Invalid Health Score":
        (
            (sensor_readings["Health_Score"] < 0)
            | (sensor_readings["Health_Score"] > 100)
        ).sum(),

    "Negative Failure Downtime":
        (failures["Downtime_Hours"] < 0).sum(),

    "Negative Maintenance Duration":
        (maintenance["Duration_Hours"] < 0).sum(),

    "Negative Work Order Hours":
        (work_orders["Actual_Hours"] < 0).sum(),

    "Negative Total Cost":
        (costs["Total_Cost"] < 0).sum()
}

pd.Series(
    business_rule_checks,
    name="Invalid_Rows"
)

Invalid Criticality              0
Invalid Asset Status             0
Invalid Load Percentage          0
Invalid Health Score             0
Negative Failure Downtime        0
Negative Maintenance Duration    0
Negative Work Order Hours        0
Negative Total Cost              0
Name: Invalid_Rows, dtype: int64

In [50]:
## Date consistency
date_checks = {
    "WO Scheduled Before Created":
        (
            work_orders["Scheduled_DateTime"]
            < work_orders["Created_DateTime"]
        ).sum(),

    "WO Completed Before Created":
        (
            work_orders["Completed_DateTime"]
            < work_orders["Created_DateTime"]
        ).sum(),

    "Maintenance Before Failure":
        (
            maintenance.loc[
                maintenance["Failure_ID"].notna()
            ]
            .merge(
                failures[
                    ["Failure_ID", "Failure_DateTime"]
                ],
                on="Failure_ID",
                how="left"
            )
            .eval(
                "Maintenance_Date < Failure_DateTime"
            )
        ).sum()
}

pd.Series(
    date_checks,
    name="Invalid_Rows"
)

WO Scheduled Before Created    3
WO Completed Before Created    0
Maintenance Before Failure     0
Name: Invalid_Rows, dtype: int64

In [51]:
expected_total = (
    costs["Labour_Cost"]
    + costs["Parts_Cost"]
    + costs["Contractor_Cost"]
    + costs["Downtime_Cost"]
    + costs["Other_Cost"]
).round(2)

cost_mismatch = (
    expected_total
    != costs["Total_Cost"].round(2)
).sum()

print(
    "Incorrect Total_Cost rows:",
    cost_mismatch
)

Incorrect Total_Cost rows: 0


In [52]:
## Cross-table completeness
print(
    "Sites:",
    len(sites)
)

print(
    "Assets:",
    len(assets)
)

print(
    "Assets with sensor readings:",
    sensor_readings["Asset_ID"].nunique()
)

print(
    "Failures:",
    len(failures)
)

print(
    "Failures linked to maintenance:",
    failures["Failure_ID"]
    .isin(
        maintenance["Failure_ID"].dropna()
    )
    .sum()
)

print(
    "Maintenance records:",
    len(maintenance)
)

print(
    "Maintenance linked to work orders:",
    maintenance["Maintenance_ID"]
    .isin(work_orders["Maintenance_ID"])
    .sum()
)

print(
    "Work orders:",
    len(work_orders)
)

print(
    "Work orders linked to costs:",
    work_orders["WorkOrder_ID"]
    .isin(costs["WorkOrder_ID"])
    .sum()
)

Sites: 3
Assets: 150
Assets with sensor readings: 150
Failures: 40
Failures linked to maintenance: 40
Maintenance records: 310
Maintenance linked to work orders: 310
Work orders: 310
Work orders linked to costs: 310


In [53]:
## Final validation result
all_checks = (
    sum(fk_checks.values())
    + sum(optional_fk_checks.values())
    + sum(business_rule_checks.values())
    + sum(date_checks.values())
    + cost_mismatch
)

if all_checks == 0:
    print("✅ PHASE 2 DATA QUALITY VALIDATION PASSED")
else:
    print(
        f"❌ VALIDATION FAILED — "
        f"{all_checks} issues found"
    )

❌ VALIDATION FAILED — 3 issues found


In [54]:
print("=== FOREIGN KEY CHECKS ===")
for k, v in fk_checks.items():
    print(k, ":", v)

print("\n=== OPTIONAL FOREIGN KEY CHECKS ===")
for k, v in optional_fk_checks.items():
    print(k, ":", v)

print("\n=== BUSINESS RULE CHECKS ===")
for k, v in business_rule_checks.items():
    print(k, ":", v)

print("\n=== DATE CHECKS ===")
for k, v in date_checks.items():
    print(k, ":", v)

print("\n=== COST CHECK ===")
print("Cost mismatch:", cost_mismatch)

=== FOREIGN KEY CHECKS ===
assets.Site_ID : 0
sensor_readings.Asset_ID : 0
failures.Asset_ID : 0
maintenance.Asset_ID : 0
work_orders.Asset_ID : 0
costs.Asset_ID : 0

=== OPTIONAL FOREIGN KEY CHECKS ===
maintenance.Failure_ID : 0
work_orders.Maintenance_ID : 0
work_orders.Failure_ID : 0
costs.WorkOrder_ID : 0
costs.Maintenance_ID : 0
costs.Failure_ID : 0

=== BUSINESS RULE CHECKS ===
Invalid Criticality : 0
Invalid Asset Status : 0
Invalid Load Percentage : 0
Invalid Health Score : 0
Negative Failure Downtime : 0
Negative Maintenance Duration : 0
Negative Work Order Hours : 0
Negative Total Cost : 0

=== DATE CHECKS ===
WO Scheduled Before Created : 3
WO Completed Before Created : 0
Maintenance Before Failure : 0

=== COST CHECK ===
Cost mismatch: 0


In [55]:
invalid_wo_dates = (
    work_orders["Scheduled_DateTime"]
    < work_orders["Created_DateTime"]
)

print("Invalid rows before fix:", invalid_wo_dates.sum())

work_orders.loc[
    invalid_wo_dates,
    "Scheduled_DateTime"
] = (
    work_orders.loc[
        invalid_wo_dates,
        "Created_DateTime"
    ]
    + pd.to_timedelta(1, unit="h")
)

print(
    "Invalid rows after fix:",
    (
        work_orders["Scheduled_DateTime"]
        < work_orders["Created_DateTime"]
    ).sum()
)

Invalid rows before fix: 3
Invalid rows after fix: 0


In [ ]:
""" work_orders.to_csv(
    "../data/work_orders.csv",
    index=False
)

print("work_orders.csv updated successfully.") """

work_orders.csv updated successfully.


In [57]:
date_checks = {
    "WO Scheduled Before Created":
        (
            work_orders["Scheduled_DateTime"]
            < work_orders["Created_DateTime"]
        ).sum(),

    "WO Completed Before Created":
        (
            work_orders["Completed_DateTime"]
            < work_orders["Created_DateTime"]
        ).sum(),

    "Maintenance Before Failure":
        (
            maintenance.loc[
                maintenance["Failure_ID"].notna()
            ]
            .merge(
                failures[
                    ["Failure_ID", "Failure_DateTime"]
                ],
                on="Failure_ID",
                how="left"
            )
            .eval(
                "Maintenance_Date < Failure_DateTime"
            )
        ).sum()
}

pd.Series(date_checks)

WO Scheduled Before Created    0
WO Completed Before Created    0
Maintenance Before Failure     0
dtype: int64

In [58]:
all_checks = (
    sum(fk_checks.values())
    + sum(optional_fk_checks.values())
    + sum(business_rule_checks.values())
    + sum(date_checks.values())
    + cost_mismatch
)

if all_checks == 0:
    print("✅ PHASE 2 DATA QUALITY VALIDATION PASSED")
else:
    print(
        f"❌ VALIDATION FAILED — "
        f"{all_checks} issues found"
    )

✅ PHASE 2 DATA QUALITY VALIDATION PASSED


#### Phase 2 Final Validation Gate

### Objective
Confirm that all synthetic operational datasets are complete, internally consistent, and ready for ingestion into Microsoft Fabric.

### Final Dataset Inventory

| Dataset | Purpose |
|---|---|
| `sites.csv` | Site master data |
| `assets.csv` | Equipment asset master |
| `sensor_readings.csv` | Hourly equipment telemetry |
| `failures.csv` | Equipment failure events |
| `maintenance.csv` | Planned and corrective maintenance |
| `work_orders.csv` | Maintenance work-order lifecycle |
| `costs.csv` | Maintenance and downtime costs |

### Validation Criteria

Phase 2 is considered complete when:

- All primary keys are unique
- No primary keys are null
- All foreign-key relationships are valid
- Sensor readings reference valid assets
- Failure events reference valid assets
- Maintenance records reference valid assets and failures
- Work orders reference valid maintenance events
- Cost records reference valid work orders
- No negative duration or cost values exist
- Health score remains between 0 and 100
- Load percentage remains between 0 and 100
- Work-order dates follow valid chronological order
- Total cost calculations are correct
- All generated datasets are successfully exported

### Final Status

**Phase 2 — Data Setup & Quality: PASSED**

The project now has a validated synthetic equipment-operations dataset ready for Microsoft Fabric ingestion and downstream analytics.